## Meteorological data processing

Average a SUMMA ensemble forcing directory into one mean forcing file. Each member folder (for example `ensemble/001/`) holds one hourly netCDF. The mean is taken at each time and HRU for the meteorological fields. `data_step`, `hruId`, and the coordinates are copied from the first member.

In [1]:
from pathlib import Path

import numpy as np
import xarray as xr

%load_ext autoreload
%autoreload 2

In [2]:
FORCING_VARIABLES = (
    "LWRadAtm",
    "SWRadAtm",
    "airpres",
    "airtemp",
    "pptrate",
    "spechum",
    "windspd",
)


def ensemble_forcing_files(ensemble_dir: Path | str) -> list[Path]:
    """Return one forcing netCDF per member directory, sorted by member id."""
    ensemble_dir = Path(ensemble_dir)
    files = sorted(path for path in ensemble_dir.glob("*/*.nc") if path.is_file())
    if not files:
        files = sorted(path for path in ensemble_dir.glob("*.nc") if path.is_file())
    if not files:
        raise FileNotFoundError(f"No ensemble forcing files in {ensemble_dir}")
    return files


def mean_ensemble_forcing(
    ensemble_dir: Path | str,
    output_path: Path | str | None = None,
) -> xr.Dataset:
    """Arithmetic mean of SUMMA ensemble forcing at each time and HRU.

    ``ensemble_dir`` contains one subdirectory per member, each with a single
    forcing netCDF. Non-finite values are left out of the mean. When
    ``output_path`` is set, the mean is written as float32 netCDF.
    """
    files = ensemble_forcing_files(ensemble_dir)
    with xr.open_dataset(files[0]) as template:
        reference_time = template["time"].values.copy()
        missing = [name for name in FORCING_VARIABLES if name not in template]
        if missing:
            raise KeyError(f"{files[0]} is missing {missing}")
        mean_ds = template.load()

    total = None
    count = None
    for path in files:
        with xr.open_dataset(path) as member:
            if not np.array_equal(member["time"].values, reference_time):
                raise ValueError(f"Time axis in {path} does not match {files[0]}")
            missing = [name for name in FORCING_VARIABLES if name not in member]
            if missing:
                raise KeyError(f"{path} is missing {missing}")
            piece = member[list(FORCING_VARIABLES)].load().astype("float64")

        valid = np.isfinite(piece)
        piece = piece.where(valid, 0.0)
        valid_count = valid.astype("int16")
        if total is None:
            total = piece
            count = valid_count
        else:
            total = total + piece
            count = count + valid_count

    averaged = total / count.where(count > 0)
    for name in FORCING_VARIABLES:
        values = averaged[name].values.astype(mean_ds[name].dtype)
        mean_ds[name] = mean_ds[name].copy(data=values)
        mean_ds[name].encoding.clear()

    mean_ds.attrs["title"] = "Ensemble-mean meteorological forcing"
    mean_ds.attrs["comment"] = (
        f"Arithmetic mean of {len(files)} members from {Path(ensemble_dir)}"
    )
    mean_ds.attrs["ensemble_members"] = len(files)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        # Input files are chunked one timestep at a time, which inflates the
        # file. Store each forcing variable in a single chunk instead.
        chunk = (int(mean_ds.sizes["time"]), int(mean_ds.sizes["hru"]))
        encoding = {
            name: {
                "dtype": "float32",
                "zlib": True,
                "complevel": 4,
                "chunksizes": chunk,
            }
            for name in FORCING_VARIABLES
        }
        mean_ds.encoding["unlimited_dims"] = set()
        mean_ds.to_netcdf(output_path, encoding=encoding, unlimited_dims=[])
        print(f"Wrote {output_path} from {len(files)} members")

    return mean_ds

## Ross ensemble mean

Members live in `forcing/ensemble/001` through `050`. The mean file is written beside that directory.

In [3]:
ensemble_dir = Path("/Users/dcasson/Data/snowda/summa/ross_lumped/forcing/ensemble")
output_path = ensemble_dir.parent / "mean" / "yrb_rf_prod_ensMean.nc"

mean_forcing = mean_ensemble_forcing(ensemble_dir, output_path)
mean_forcing

Wrote /Users/dcasson/Data/snowda/summa/ross_lumped/forcing/mean/yrb_rf_prod_ensMean.nc from 50 members


<xarray.Dataset> Size: 14MB
Dimensions:    (time: 394440, hru: 1)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1980-01-01 ... 2024-12-29T23:00:00
  * hru        (hru) float64 8B 0.0
Data variables:
    data_step  int32 4B 3600
    LWRadAtm   (time, hru) float32 2MB 228.9 225.3 221.2 ... 238.2 238.0 237.6
    SWRadAtm   (time, hru) float32 2MB 0.0 0.0 0.0 0.0 ... 6.976 3.228 0.01414
    airpres    (time, hru) float32 2MB 8.757e+04 8.752e+04 ... 8.764e+04
    airtemp    (time, hru) float32 2MB 263.3 262.3 261.1 ... 265.0 265.0 264.9
    hruId      (hru) int32 4B 1
    pptrate    (time, hru) float32 2MB 6.438e-07 6.438e-07 ... 6.438e-07
    spechum    (time, hru) float32 2MB 0.0002672 0.0002673 ... 0.001395 0.001395
    windspd    (time, hru) float32 2MB 1.306 1.304 1.239 ... 0.9604 1.189 1.191
Attributes: (12/36)
    CDI:                      Climate Data Interface version 1.9.9 (https://m...
    Conventions:              CF-1.6
    conventions:              1.6
    title:                    Ensemble-mean meteorological forcing
    comment:                  Arithmetic mean of 50 members from /Users/dcass...
    period_ending:            False
    ...                       ...
    Author:                   Created by SUMMA workflow scripts
    NCO:                      netCDF Operators version 4.9.3 (Homepage = http...
    history:                  Tue Oct 27 16:38:03 2026: ncap2 -O -s data_step...
    History:                  Tue Oct 27 16:33:25 2026: ncrename -O -v .LWrad...
    CDO:                      Climate Data Operators version 1.9.9 (https://m...
    ensemble_members:         50